# Probability Calibration: Platt Scaling

This notebook demonstrates Platt Scaling, a post-hoc calibration method that fits a sigmoid function to map raw model outputs to calibrated probabilities.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score, brier_score_loss, log_loss
from sklearn.metrics import calibration_curve

np.random.seed(42)

## 1. Load Data

In [ ]:
iris = load_iris()
X, y = iris.data, iris.target

# For binary classification: Setosa vs. others
y_binary = (y == 0).astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_binary, test_size=0.3, random_state=42, stratify=y_binary
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Class distribution - Train: {np.bincount(y_train)}, Test: {np.bincount(y_test)}")

In [ ]:
# SVM without probability calibration
svm = SVC(kernel='rbf', random_state=42, probability=False)
svm.fit(X_train, y_train)

# Get decision function scores (raw outputs)
scores_train = svm.decision_function(X_train)
scores_test = svm.decision_function(X_test)

y_pred = svm.predict(X_test)
acc = accuracy_score(y_test, y_pred)

print(f"Uncalibrated SVM Accuracy: {acc:.4f}")
print(f"Decision function range (train): [{scores_train.min():.2f}, {scores_train.max():.2f}]")
print(f"Decision function range (test): [{scores_test.min():.2f}, {scores_test.max():.2f}]")

## 3. Apply Platt Scaling Calibration

In [ ]:
# Platt scaling calibration
calibrator = CalibratedClassifierCV(SVC(kernel='rbf', random_state=42, probability=False), 
                                    method='sigmoid', cv=5)
calibrator.fit(X_train, y_train)

# Get calibrated probabilities
proba_cal = calibrator.predict_proba(X_test)
y_pred_cal = calibrator.predict(X_test)
acc_cal = accuracy_score(y_test, y_pred_cal)

print(f"Calibrated SVM Accuracy: {acc_cal:.4f}")
print(f"Probability range: [{proba_cal.min():.4f}, {proba_cal.max():.4f}]")

## 4. Evaluate Calibration Quality

In [ ]:
# Get probabilities from native SVM (for comparison)
svm_with_proba = SVC(kernel='rbf', random_state=42, probability=True)
svm_with_proba.fit(X_train, y_train)
proba_uncal = svm_with_proba.predict_proba(X_test)

# Calibration metrics
brier_uncal = brier_score_loss(y_test, proba_uncal[:, 1])
brier_cal = brier_score_loss(y_test, proba_cal[:, 1])

logloss_uncal = log_loss(y_test, proba_uncal[:, 1])
logloss_cal = log_loss(y_test, proba_cal[:, 1])

print("\n=== CALIBRATION METRICS ===")
print(f"Brier Score (lower is better):")
print(f"  Uncalibrated: {brier_uncal:.4f}")
print(f"  Calibrated:   {brier_cal:.4f}")
print(f"  Improvement:  {(brier_uncal - brier_cal):.4f}")

print(f"\nLog Loss (lower is better):")
print(f"  Uncalibrated: {logloss_uncal:.4f}")
print(f"  Calibrated:   {logloss_cal:.4f}")
print(f"  Improvement:  {(logloss_uncal - logloss_cal):.4f}")

## 5. Calibration Curve

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Uncalibrated calibration curve
frac_pos_uncal, mean_pred_uncal = calibration_curve(y_test, proba_uncal[:, 1], n_bins=10)
ax1.plot(mean_pred_uncal, frac_pos_uncal, 'o-', linewidth=2, markersize=8, label='SVM')
ax1.plot([0, 1], [0, 1], 'k--', label='Perfectly calibrated')
ax1.set_xlabel('Mean predicted probability')
ax1.set_ylabel('Fraction of positives')
ax1.set_title('Uncalibrated SVM')
ax1.legend()
ax1.grid(alpha=0.3)

# Calibrated calibration curve
frac_pos_cal, mean_pred_cal = calibration_curve(y_test, proba_cal[:, 1], n_bins=10)
ax2.plot(mean_pred_cal, frac_pos_cal, 'o-', linewidth=2, markersize=8, label='Calibrated (Platt)')
ax2.plot([0, 1], [0, 1], 'k--', label='Perfectly calibrated')
ax2.set_xlabel('Mean predicted probability')
ax2.set_ylabel('Fraction of positives')
ax2.set_title('Platt-Scaled SVM')
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Probability distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(proba_uncal[y_test==0, 1], bins=20, alpha=0.7, label='Class 0', color='blue')
axes[0].hist(proba_uncal[y_test==1, 1], bins=20, alpha=0.7, label='Class 1', color='red')
axes[0].set_xlabel('Predicted Probability (Class 1)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Uncalibrated SVM Probabilities')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].hist(proba_cal[y_test==0, 1], bins=20, alpha=0.7, label='Class 0', color='blue')
axes[1].hist(proba_cal[y_test==1, 1], bins=20, alpha=0.7, label='Class 1', color='red')
axes[1].set_xlabel('Predicted Probability (Class 1)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Calibrated (Platt Scaling) SVM Probabilities')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
## 6. Key Insights

**Platt Scaling:**
- Fits sigmoid: P(y=1|x) = 1 / (1 + exp(Ax + B))
- Requires separate calibration data
- Works well when model outputs are monotonic with class probability
- Computationally cheap (just 2 parameters)
- Often improves probability estimates for SVM